# Visual Table Assistant — Dataset Preparation

## Purpose

This notebook builds the final YOLO dataset for Phase 1 from the raw zips
stored on Google Drive. Output is a flat layout under
`datasets/table_assistant_yolo/` plus a manifest, supporting reports, and a
single ZIP package (`datasets/table_assistant_yolo_package.zip`) ready to be
tracked with DVC.

The train/val/test split is **not** generated here. It is regenerated in
`02_training_colab.ipynb` after the package is pulled, so the split files
always reflect the runtime paths of the active environment.

Training does not happen here either. Once this notebook finishes
successfully and the package has been pushed via DVC, training runs in
`02_training_colab.ipynb`.

## Detection classes (frozen)

The seven YOLO classes used end-to-end by the project are defined in
`configs/classes.yaml`. They are reproduced here for quick reference:

| ID | Class |
|---:|---|
| 0 | food |
| 1 | cup |
| 2 | bottle |
| 3 | plate |
| 4 | spoon |
| 5 | fork |
| 6 | knife |

`food` is generic. The system does not aim to identify the type of food.

## 1. Environment Setup

Verify Python and clone (or update) the project repository under `/content/`.
Cells use absolute paths because Colab sessions can disconnect mid-run, and a
`%cd` from a previous cell does not survive the reconnect; absolute paths
make every cell safe to re-run independently.

### 1.1 Checking Python Version

In [ ]:
import sys
import platform

print("Python version:", sys.version)
print("Platform:", platform.platform())

### 1.2 Project Path Variables

In [ ]:
REPO_URL = "https://github.com/LucasGVallejos/iaa-visual-table-assistant"
WORKSPACE_DIR = "/content"
PROJECT_DIR = "iaa-visual-table-assistant"
PROJECT_PATH = f"{WORKSPACE_DIR}/{PROJECT_DIR}"

### 1.3 Cloning GitHub Repository

In [ ]:
import os

%cd {WORKSPACE_DIR}

if not os.path.exists(PROJECT_PATH):
    !git clone {REPO_URL} {PROJECT_PATH}
else:
    print("Repository already exists. Pulling latest changes...")
    !git -C {PROJECT_PATH} pull

%cd {PROJECT_PATH}

### 1.4 Installing Dependencies

In [ ]:
!pip install -r requirements.txt

### 1.5 Sanity Check on Installed Libraries

In [ ]:
import torch
import ultralytics
import cv2
import mlflow
import onnx
import onnxruntime
import dvc
import numpy as np
import pandas as pd

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

print("OpenCV:", cv2.__version__)
print("ONNX Runtime:", onnxruntime.__version__)

## 2. Raw Dataset Setup from Google Drive

Mount Google Drive, extract the raw zips into the repo, inspect their structure,
and run a visual bounding-box sanity check before any conversion. All logic
lives in versioned scripts under `src/data/raw_setup/`.

**Expected Drive layout:**
```
MyDrive/iaa-table-assistant/
  raw_datasets/
    open_images/          # zip(s) from download_open_images_subset.py
    uec_food_256/         # zip(s) from UEC FOOD-256
```

### 2.1 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### 2.2 Extract and Inspect Raw Datasets

Locates the raw zips on Drive, extracts them into
`datasets/raw_datasets/` inside the repo, and prints a structural summary so
you can spot missing folders or unexpected layouts before continuing.

In [ ]:
!python -m src.data.raw_setup.setup_colab_raw_datasets

### 2.3 Visual Bounding Box Sanity Check (Raw)

Renders one sample image per source with its raw bounding boxes drawn on top.
Confirms that COCO and UEC bbox parsing is correct **before** any conversion
to YOLO format.

Outputs: `outputs/bbox_checks/{open_images_sample,uec_food_sample}.png`.

In [ ]:
!python -m src.data.raw_setup.visualize_raw_bboxes

In [ ]:
from IPython.display import Image, display

display(Image("outputs/bbox_checks/open_images_sample.png"))
display(Image("outputs/bbox_checks/uec_food_sample.png"))

## 3. Per-Source Conversion to YOLO Staging

Each source dataset is converted to YOLO format independently into its own
staging directory under `datasets/_staging/`. Per-source staging avoids
filename collisions between sources and lets us validate each one in
isolation before merging.

Output:
```
datasets/_staging/
  ├── uec_food/{images,labels}/
  └── open_images/{images,labels}/
reports/skipped_images/
  ├── uec_food_256.csv
  └── open_images.csv
```

### 3.1 Convert UEC FOOD-256 to YOLO

Walks the per-category folders, parses each `bb_info.txt`, converts VOC-style
bboxes to YOLO normalized xywh, and writes the staging copy. Every UEC bbox
becomes class id 0 (food).

In [ ]:
!python -m src.data.conversion.convert_uec_food_to_yolo

### 3.2 Convert Open Images to YOLO

Reads the COCO export, maps source category names to YOLO class IDs via
`configs/label_mapping.yaml`, converts COCO `[x, y, w, h]` to YOLO normalized
`[cx, cy, w, h]`, and writes the staging copy. Categories not present in
`label_mapping.yaml` are dropped.

In [ ]:
!python -m src.data.conversion.convert_open_images_to_yolo

### 3.3 Sanity Checks on Staging

Two complementary checks once both stagings exist:

1. **Visual mapping check**: render random samples per class with their
   converted YOLO boxes, so you can eyeball that `cup` boxes really wrap cups,
   `knife` wraps knives, etc. Output goes under
   `outputs/staging_bbox_checks/seed_<NN>/`.
2. **Class distribution analysis**: aggregate counts per class, share of total,
   imbalance ratio, and bbox size stats. Result is printed to stdout and
   persisted as `reports/class_distribution_<timestamp>.json` (gitignored).

In [ ]:
!python -m src.data.validation.visualize_yolo_mapping --seed 7 --samples-per-class 5

In [ ]:
!python -m src.data.validation.analyze_class_distribution

## 4. Merge into Final YOLO Dataset

`prepare_dataset.py` merges both staging dirs into a single flat layout
`datasets/table_assistant_yolo/{images,labels}/`. Each pair is renamed to
`<NNNNNNNN>_<class_a>_<class_b>...jpg` (sequence + sorted class names) so the
filename encodes content. Provenance lives in `reports/dataset_manifest.csv`.
After the merge the staging dirs are deleted.

The train/val/test split is intentionally **not** generated here. The split
files reference absolute paths and would be invalid the moment the dataset is
moved (e.g. pulled into a different Colab session). Splits are produced in
`02_training_colab.ipynb` from the unzipped package, with seed 42 and
stratification by the rarest class per image.

### 4.1 Merge Per-Source Staging into the Final Layout

In [ ]:
!python -m src.data.preparation.prepare_dataset

### 4.2 Refresh `reports/dataset_notes.md`

Recomputes totals (images, label files, boxes) and per-class image and box
counts from the merged dataset, then rewrites the auto-managed
`## Class Distribution` section in `reports/dataset_notes.md` between the
`<!-- BEGIN_SPLIT_DISTRIBUTION -->` and `<!-- END_SPLIT_DISTRIBUTION -->`
markers. The split notebook later overwrites the same section with
split-aware tables.

In [ ]:
!python -m src.data.preparation.update_dataset_notes

## 5. Package Final YOLO Dataset for DVC

Bundle the trainable dataset together with its provenance metadata into a
single `.zip` so DVC has exactly one artifact to track.

Final ZIP layout:
```
table_assistant_yolo_package/
├── table_assistant_yolo/
│   ├── images/
│   └── labels/
└── metadata/
    ├── dataset_manifest.csv
    ├── dataset_notes.md
    ├── dataset_package_metadata.json
    ├── class_distribution_<latest>.json
    └── skipped_images/
        ├── open_images.csv
        └── uec_food_256.csv
```

Splits, visual checks, configs, raw and staging datasets are deliberately
excluded. The split text files would be invalidated by paths changing across
environments; the rest is reproducible from code.

### 5.1 Build the Package Directory

In [ ]:
import shutil
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_PATH)
DATASET_DIR = PROJECT_ROOT / "datasets" / "table_assistant_yolo"
REPORTS_DIR = PROJECT_ROOT / "reports"
PACKAGE_NAME = "table_assistant_yolo_package"
PACKAGE_DIR = PROJECT_ROOT / "datasets" / PACKAGE_NAME
PACKAGE_ZIP = PROJECT_ROOT / "datasets" / f"{PACKAGE_NAME}.zip"

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

# Dataset payload.
shutil.copytree(DATASET_DIR, PACKAGE_DIR / "table_assistant_yolo")

# Metadata payload.
metadata_dir = PACKAGE_DIR / "metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)

manifest_src = REPORTS_DIR / "dataset_manifest.csv"
if not manifest_src.exists():
    raise FileNotFoundError(
        f"Required manifest missing: {manifest_src}. "
        "Re-run prepare_dataset.py before packaging."
    )
shutil.copy2(manifest_src, metadata_dir / manifest_src.name)

notes_src = REPORTS_DIR / "dataset_notes.md"
if notes_src.exists():
    shutil.copy2(notes_src, metadata_dir / notes_src.name)
else:
    print(f"[WARN] {notes_src} not found; skipping.")

# Latest class_distribution_*.json by mtime.
distributions = sorted(
    REPORTS_DIR.glob("class_distribution_*.json"),
    key=lambda p: p.stat().st_mtime,
)
if distributions:
    latest = distributions[-1]
    shutil.copy2(latest, metadata_dir / latest.name)
    print(f"Included class distribution: {latest.name}")
else:
    print(
        "[WARN] No reports/class_distribution_*.json found; skipping. "
        "Re-run analyze_class_distribution.py to refresh it."
    )

# Per-source skipped_images CSVs (warn but do not fail when missing).
skipped_src_dir = REPORTS_DIR / "skipped_images"
skipped_dst_dir = metadata_dir / "skipped_images"
skipped_dst_dir.mkdir(parents=True, exist_ok=True)
for name in ("open_images.csv", "uec_food_256.csv"):
    src = skipped_src_dir / name
    if src.exists():
        shutil.copy2(src, skipped_dst_dir / name)
    else:
        print(f"[WARN] {src} not found; skipping.")

print(f"Built package directory: {PACKAGE_DIR}")

### 5.2 Generate `dataset_package_metadata.json`

Minimal manifest describing the package itself: name, layout, totals, and a
pointer to where the operative split is generated.

In [ ]:
import json
from datetime import datetime, timezone

images_dir = PACKAGE_DIR / "table_assistant_yolo" / "images"
labels_dir = PACKAGE_DIR / "table_assistant_yolo" / "labels"

total_images = sum(1 for p in images_dir.iterdir() if p.suffix.lower() == ".jpg")
total_labels = sum(1 for p in labels_dir.iterdir() if p.suffix.lower() == ".txt")

total_boxes = 0
for p in labels_dir.iterdir():
    if p.suffix.lower() != ".txt":
        continue
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                total_boxes += 1

metadata = {
    "package_name": PACKAGE_NAME,
    "dataset_name": "table_assistant_yolo",
    "layout": "flat_yolo",
    "total_images": total_images,
    "total_labels": total_labels,
    "total_boxes": total_boxes,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "split_generated_in": "02_training_colab.ipynb",
    "split_strategy": "rarest_class_per_image",
    "split_seed": 42,
}

metadata_path = PACKAGE_DIR / "metadata" / "dataset_package_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

### 5.3 Validate the Package

Cheap structural checks before zipping: dataset is not empty, every image
has a matching label, manifest is present.

In [ ]:
errors = []

if total_images == 0:
    errors.append("No images in package (table_assistant_yolo/images/ is empty).")
if total_labels == 0:
    errors.append("No label files in package (table_assistant_yolo/labels/ is empty).")
if total_images != total_labels:
    errors.append(
        f"Image/label count mismatch: {total_images} images vs "
        f"{total_labels} label files."
    )
if not (PACKAGE_DIR / "metadata" / "dataset_manifest.csv").exists():
    errors.append("Missing metadata/dataset_manifest.csv.")

if errors:
    raise RuntimeError(
        "Package validation failed:\n  - " + "\n  - ".join(errors)
    )

print("Package validation OK.")
print(f"  images: {total_images}")
print(f"  labels: {total_labels}")
print(f"  boxes:  {total_boxes}")

### 5.4 Create the ZIP

Final artifact for DVC: `datasets/table_assistant_yolo_package.zip`. Tracking a
single file keeps DVC fast and produces deterministic content hashes.

In [ ]:
if PACKAGE_ZIP.exists():
    PACKAGE_ZIP.unlink()

# shutil.make_archive needs the base name without extension and the parent dir
# of the directory we want to zip, so ZIP entries are relative to that parent.
archive_base = PACKAGE_ZIP.with_suffix("")
shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=str(PACKAGE_DIR.parent),
    base_dir=PACKAGE_DIR.name,
)

size_gb = PACKAGE_ZIP.stat().st_size / (1024 ** 3)
print(f"Created {PACKAGE_ZIP}")
print(f"  images:   {total_images}")
print(f"  labels:   {total_labels}")
print(f"  zip size: {size_gb:.2f} GB")

## 6. DVC tracking — run locally

DVC pushes are gated to local execution to keep credentials out of Colab.
After this notebook finishes:

1. **Download** `datasets/table_assistant_yolo_package.zip` from this Colab
   session (or copy it to Drive) and place it under
   `datasets/table_assistant_yolo_package.zip` in your local checkout.
2. **Track and push** from your local shell:
   ```bash
   dvc add datasets/table_assistant_yolo_package.zip
   git add datasets/table_assistant_yolo_package.zip.dvc datasets/.gitignore
   git commit -m "track table_assistant_yolo_package.zip with DVC"
   dvc push
   git push
   ```

**Commit to git:** `datasets/table_assistant_yolo_package.zip.dvc`,
`datasets/.gitignore`.

**Do not commit to git:** the ZIP itself, the unpacked
`datasets/table_assistant_yolo_package/` directory, or the merged
`datasets/table_assistant_yolo/` directory. DVC tracks the ZIP; the rest is
either intermediate or reproducible from code.

Once pushed, `02_training_colab.ipynb` pulls the ZIP, unpacks it, regenerates
the train/val/test split with seed 42, and starts training.

In [ ]:
from google.colab import files
import os

zip_file_path = '/content/iaa-visual-table-assistant/datasets/table_assistant_yolo_package.zip'

if os.path.exists(zip_file_path):
    print(f"Starting download {zip_file_path}...")
    files.download(zip_file_path)
    print("Download requested. Please check your downloads. If the download doesn't start automatically, the file may still be processing and will begin shortly.")
else:
    print(f"Error: The Zip file isn't in {zip_file_path}.")